# 01 - Quantum Fundamentals: Superposition, Gates & Entanglement

**Module 1 - Quantum Circuit Design Platform**

This notebook demonstrates the three concepts a classical bit cannot express:

1. **Superposition** - a qubit can be in a weighted combination of |0> and |1> at once. The **Hadamard (H)** gate creates an equal superposition.
2. **Gates** - unitary operations that rotate the qubit state. H puts a qubit into superposition; **CNOT (CX)** flips a target qubit conditioned on a control qubit.
3. **Entanglement** - after H on qubit 0 then CNOT(0->1), the two qubits form a **Bell state** `(|00> + |11>)/sqrt(2)`. Measuring one instantly determines the other: we only ever see `00` or `11`, never `01` or `10`.

**Classical vs quantum:** a classical 2-bit register holds exactly one of {00,01,10,11}. The Bell state holds `00` and `11` simultaneously until measured, and the two bits are correlated in a way no classical register can reproduce.

We use the platform's own `build_bell_state_circuit()` and `run_circuit()` from `backend/app/quantum/sanity_check.py` running on Qiskit's `AerSimulator`.

In [1]:
# Make the platform's backend package importable from this notebook.
# The notebooks call the SAME code the FastAPI app runs, so results here
# and results in the deployed platform are guaranteed identical.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'backend')))
print('Backend path wired. Python', sys.version.split()[0])

Backend path wired. Python 3.14.4


In [2]:
from app.quantum.sanity_check import build_bell_state_circuit, run_circuit

qc = build_bell_state_circuit()
print('Bell-state circuit:')
print(qc.draw(output='text'))

Bell-state circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


### Run the simulation (1024 shots)

We expect roughly a 50/50 split between `00` and `11`, and essentially zero `01`/`10`. That absence of `01`/`10` is the signature of entanglement.

In [3]:
counts = run_circuit(qc, shots=1024)
print('Measurement counts:', counts)

total = sum(counts.values())
for state in ['00', '01', '10', '11']:
    c = counts.get(state, 0)
    print(f'  {state}: {c:4d}  ({100*c/total:5.1f}%)')

Measurement counts: {'11': 526, '00': 498}
  00:  498  ( 48.6%)
  01:    0  (  0.0%)
  10:    0  (  0.0%)
  11:  526  ( 51.4%)


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

states = sorted(counts)
plt.figure(figsize=(5,3))
plt.bar(states, [counts[s] for s in states], color='#6c5ce7')
plt.title('Bell state measurement distribution (1024 shots)')
plt.xlabel('Measured bitstring'); plt.ylabel('Counts')
plt.tight_layout(); plt.show()

/tmp/ipykernel_29893/3553457936.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What this proves:** the ~50/50 `00`/`11` split with near-zero `01`/`10` is produced by a real statevector simulation, not hardcoded numbers. This is the foundation every other module builds on.